# Lab Assignment: Exploring U-Net Variants for Image Segmentation (Carvana Dataset)

**Total Marks: 10**  
**Course:** Deep Learning / Computer Vision  
**Topic:** Semantic Segmentation with U-Net Variants

In this lab, you will:
- Implement and train a **U-Net** on the **Carvana Image Masking Dataset**.
- Use **BCE loss** for training and **Dice Score** for evaluation.
- Experiment with architectural variations to study their effects on performance.

---

## 🧾 1. Dataset Setup: Carvana Image Masking Dataset

The Carvana dataset contains car images and corresponding binary masks for car segmentation.

**Download Instructions:**
1. Go to [Carvana Image Masking Challenge on Kaggle](https://www.kaggle.com/competitions/carvana-image-masking-challenge/data).
2. Download and extract files into the following directory structure:
   ```
   data/
     carvana/
       train/
         <image_id>.jpg
       train_masks/
         <image_id>_mask.png
   ```
3. Place this `data` folder in the same directory as this notebook.


## 2. Imports and Utility Functions

In [17]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 3. Define U-Net (with Residual Connections, No BatchNorm)

In [18]:
class ResidualDoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.res = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.conv(x) + self.res(x)


class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        self.inc = ResidualDoubleConv(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ResidualDoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ResidualDoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ResidualDoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), ResidualDoubleConv(512, 1024))

        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv1 = ResidualDoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv2 = ResidualDoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = ResidualDoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv4 = ResidualDoubleConv(128, 64)

        self.outc = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = torch.cat([x4, x], dim=1)
        x = self.conv1(x)
        x = self.up2(x)
        x = torch.cat([x3, x], dim=1)
        x = self.conv2(x)
        x = self.up3(x)
        x = torch.cat([x2, x], dim=1)
        x = self.conv3(x)
        x = self.up4(x)
        x = torch.cat([x1, x], dim=1)
        x = self.conv4(x)
        return self.outc(x)

## 4. Dice Score Metric and BCE Loss

In [19]:
def dice_score(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2 * intersection + smooth) / (union + smooth)
    return dice.mean()

## 5. Carvana Dataset Loader

In [20]:
class CarvanaDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name.replace('.jpg', '_mask.gif'))
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        mask = (mask > 0.5).float()
        return image, mask

## 6. Training and Evaluation Functions

In [21]:
def train_model(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for img, mask in tqdm(loader, desc='Training', leave=False):
        img, mask = img.to(device), mask.to(device)
        pred = model(img)
        loss = criterion(pred, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_model(model, loader, device):
    model.eval()
    dice_total = 0
    with torch.no_grad():
        for img, mask in loader:
            img, mask = img.to(device), mask.to(device)
            pred = model(img)
            dice_total += dice_score(pred, mask).item()
    return dice_total / len(loader)

## 7. Train the Model on Carvana Dataset (70:10:20 split)

In [22]:
import os

# Define the paths to the training images and masks
train_image_dir = 'data/carvana/train/train'
train_mask_dir = 'data/carvana/train_masks/train_masks'

# Check if the directories exist before listing files
if os.path.exists(train_image_dir) and os.path.exists(train_mask_dir):
    # Get the list of training image and mask filenames
    train_image_filenames = sorted([f for f in os.listdir(train_image_dir) if f.endswith('.jpg')])
    train_mask_filenames = sorted([f for f in os.listdir(train_mask_dir) if f.endswith('_mask.gif')])

    print(f"Found {len(train_image_filenames)} training images and {len(train_mask_filenames)} training masks.")

    # Display the first 5 image and mask filenames
    print("\nFirst 5 training image filenames:")
    for filename in train_image_filenames[:5]:
        print(filename)

    print("\nFirst 5 training mask filenames:")
    for filename in train_mask_filenames[:5]:
        print(filename)
else:
    print(f"Error: Directories '{train_image_dir}' or '{train_mask_dir}' not found. Please run the previous cell to extract the data.")

Found 5088 training images and 5088 training masks.

First 5 training image filenames:
00087a6bd4dc_01.jpg
00087a6bd4dc_02.jpg
00087a6bd4dc_03.jpg
00087a6bd4dc_04.jpg
00087a6bd4dc_05.jpg

First 5 training mask filenames:
00087a6bd4dc_01_mask.gif
00087a6bd4dc_02_mask.gif
00087a6bd4dc_03_mask.gif
00087a6bd4dc_04_mask.gif
00087a6bd4dc_05_mask.gif


In [23]:
# Define transformations for the images and masks
img_transform = transforms.Compose([
    transforms.Resize((192, 192)),
    transforms.ToTensor()
])

mask_transform = transforms.Compose([
    transforms.Resize((192, 192)),
    transforms.ToTensor()
])

# Create the dataset
full_dataset = CarvanaDataset(train_image_dir, train_mask_dir, transform=img_transform)

# Split the dataset (70% train, 10% val, 20% test)
dataset_size = len(full_dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size

generator1 = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size], generator=generator1)

# Create data loaders
batch_size = 16 # You can adjust this
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Dataset split into {len(train_dataset)} training, {len(val_dataset)} validation, and {len(test_dataset)} test samples.")
print(f"Training DataLoader has {len(train_loader)} batches.")
print(f"Validation DataLoader has {len(val_loader)} batches.")
print(f"Test DataLoader has {len(test_loader)} batches.")

Dataset split into 3561 training, 508 validation, and 1019 test samples.
Training DataLoader has 223 batches.
Validation DataLoader has 32 batches.
Test DataLoader has 64 batches.


In [25]:
model = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(3):
    loss = train_model(model, train_loader, optimizer, criterion, device)
    val_dice = eval_model(model, val_loader, device)
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Val Dice={val_dice:.4f}')


Epoch 1: Loss=0.1733, Val Dice=0.9290


Epoch 2: Loss=0.0493, Val Dice=0.9598


Epoch 3: Loss=0.0297, Val Dice=0.9757


In [26]:
test_dice = eval_model(model, test_loader, device)
print(f'Final Test Dice Score: {test_dice:.4f}')

Final Test Dice Score: 0.9751


---
### Modifications to reduce computation time while minimising performance loss
- Reduce image resolution to 192x192
- num_workers=4, pin_memory=True in train loader to reduce CPU bottleneck
- Reduced epochs to 3
---

## 8. Lab Questions (10 Marks)

| No. | Experiment | Task | Marks |
|-----|-------------|------|-------|
| **1** | **Removing Residual Connections** | Modify `ResidualDoubleConv` → remove skip addition. Re-train and report Dice score. What happens to performance? | 2 |
| **2** | **Adding Batch Normalization** | Add `nn.BatchNorm2d` after each conv. Compare convergence and Dice. | 2 |
| **3** | **Adding Dice Loss (with BatchNorm)** | Use `Dice loss (`(1 - dice)`). Observe Change. | 3 |
| **4** | **Using VGG-U-Net** | Replace this model with [VGG-U-Net](https://github.com/asanakoy/kaggle_carvana_segmentation/blob/master/asanakoy/vgg_unet.py). Compare parameter count, training speed, and Dice score. | 3 |

📄 **Submission:** Include your code modifications, result loss plots, and a short analysis (3–5 lines) for each experiment.

---
### Q8.1 Removing Residual Connections

In [27]:
class DoubleConv(nn.Module):
    """
    Module designed for 2 layer convolution, without residual connections for replacement in Q8.1
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [28]:
class UNet_without_res(nn.Module):
    """
    Updated UNet architecture as per Q8.1, removing res connections
    """
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv4 = DoubleConv(128, 64)

        self.outc = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = torch.cat([x4, x], dim=1)
        x = self.conv1(x)
        x = self.up2(x)
        x = torch.cat([x3, x], dim=1)
        x = self.conv2(x)
        x = self.up3(x)
        x = torch.cat([x2, x], dim=1)
        x = self.conv3(x)
        x = self.up4(x)
        x = torch.cat([x1, x], dim=1)
        x = self.conv4(x)
        return self.outc(x)

In [29]:
model = UNet_without_res().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(3):
    loss = train_model(model, train_loader, optimizer, criterion, device)
    val_dice = eval_model(model, val_loader, device)
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Val Dice={val_dice:.4f}')

Epoch 1: Loss=0.2544, Val Dice=0.9351


Epoch 2: Loss=0.0448, Val Dice=0.9676


Epoch 3: Loss=0.0281, Val Dice=0.9760


In [30]:
test_dice = eval_model(model, test_loader, device)
print(f'Final Test Dice Score: {test_dice:.4f}')

Final Test Dice Score: 0.9758


#### Analysis
- With residual connections
    - Epoch 3: Loss=0.0297, Val Dice=0.9757, Final Test Dice Score: 0.9751

- without residual connections
    - Epoch 3: Loss=0.0281, Val Dice=0.9760, Final Test Dice Score: 0.9758

Expected drop in performance, but negligible changes, included within margin of error
Slower convergence observed in loss change during epochs

---


### Q8.2 Removing residual connections, and adding batch normalisation

In [31]:
class DoubleConv_BN(nn.Module):
    """
    Modifed Conv without residual connections but with batch normalisation
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [32]:
class UNet_without_res_with_BN(nn.Module):
    """
    Updated UNet architecture as per Q8.2, removing res connections, and adding batch normalisation
    """
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        self.inc = DoubleConv_BN(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv_BN(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv_BN(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv_BN(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv_BN(512, 1024))

        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.conv1 = DoubleConv_BN(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv2 = DoubleConv_BN(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv_BN(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv4 = DoubleConv_BN(128, 64)

        self.outc = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5)
        x = torch.cat([x4, x], dim=1)
        x = self.conv1(x)
        x = self.up2(x)
        x = torch.cat([x3, x], dim=1)
        x = self.conv2(x)
        x = self.up3(x)
        x = torch.cat([x2, x], dim=1)
        x = self.conv3(x)
        x = self.up4(x)
        x = torch.cat([x1, x], dim=1)
        x = self.conv4(x)
        return self.outc(x)

In [37]:
model = UNet_without_res_with_BN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(3):
    loss = train_model(model, train_loader, optimizer, criterion, device)
    val_dice = eval_model(model, val_loader, device)
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Val Dice={val_dice:.4f}')

Epoch 1: Loss=0.2624, Val Dice=0.9812


Epoch 2: Loss=0.1618, Val Dice=0.9830


Epoch 3: Loss=0.1138, Val Dice=0.9887


In [38]:
test_dice = eval_model(model, test_loader, device)
print(f'Final Test Dice Score: {test_dice:.4f}')

Final Test Dice Score: 0.9886


- With residual connections
    - Epoch 3: Loss=0.0297, Val Dice=0.9757, Final Test Dice Score: 0.9751

- Without residual connections, with Batch normalisation
    - Epoch 3: Loss=0.1138, Val Dice=0.9887, Final Test Dice Score: 0.9886

Signficnt improvement in dice score, which is not reflected in loss, primarily due to better IoU consolidation.

---

### Q8.3 adding dice loss, retaining batch norm

In [33]:
def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    return 1 - (2 * intersection + smooth) / (union + smooth)

criterion = lambda pred, mask: 0.5 * nn.BCEWithLogitsLoss()(pred, mask) + 0.5 * dice_loss(pred, mask)

model = UNet_without_res_with_BN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)


for epoch in range(3):
    loss = train_model(model, train_loader, optimizer, criterion, device)
    val_dice = eval_model(model, val_loader, device)
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Val Dice={val_dice:.4f}')

Epoch 1: Loss=0.2631, Val Dice=0.9830


Epoch 2: Loss=0.1639, Val Dice=0.9854


Epoch 3: Loss=0.1138, Val Dice=0.9872


In [34]:
test_dice = eval_model(model, test_loader, device)
print(f'Final Test Dice Score: {test_dice:.4f}')

Final Test Dice Score: 0.9871


#### Analysis
- With residual connections
    - Epoch 3: Loss=0.0297, Val Dice=0.9757, Final Test Dice Score: 0.9751

- Without residual connections, and with batch norm and dice loss
    - Epoch 3: Loss=0.1138, Val Dice=0.9872, Final Test Dice Score: 0.9871

Significant comparative boost in dice score, while increased loss values
Optimises IoU directly, which is estabilished in improvement on test dice scores, but not present in direct loss calculations.

Adding dice loss as critera resulted in negligible changes

---

### Q8.4 Swapping UNET with VGG-Unet

In [66]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

class VGGUNet(nn.Module):
    """
    VGG-U-Net implementation using VGG16 as encoder for Q8 4th section
    Based on: https://github.com/asanakoy/kaggle_carvana_segmentation
    """
    def __init__(self, n_classes=1, pretrained=True):
        super().__init__()
        
        # Load pretrained VGG16
        vgg = models.vgg16(pretrained=pretrained)
        
        # Encoder (VGG16 feature extractor)
        self.conv1 = nn.Sequential(*list(vgg.features.children())[:4])
        self.conv2 = nn.Sequential(*list(vgg.features.children())[4:9])
        self.conv3 = nn.Sequential(*list(vgg.features.children())[9:16])
        self.conv4 = nn.Sequential(*list(vgg.features.children())[16:23])
        self.conv5 = nn.Sequential(*list(vgg.features.children())[23:30]) 
        
        # Center/Bridge
        self.center = self._make_decoder_block(512, 512)
        
        # Decoder (Upsampling path)
        self.dec5 = self._make_decoder_block(1024, 512)
        self.dec4 = self._make_decoder_block(1024, 512)
        self.dec3 = self._make_decoder_block(768, 256)
        self.dec2 = self._make_decoder_block(384, 128)
        self.dec1 = self._make_decoder_block(192, 64)
        
        # Final output layer
        self.final = nn.Conv2d(64, n_classes, kernel_size=1)
        
    def _make_decoder_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        conv1 = self.conv1(x)
        conv2 = self.conv2(conv1)
        conv3 = self.conv3(conv2)
        conv4 = self.conv4(conv3)
        conv5 = self.conv5(conv4)
        
        center = self.center(conv5)
        
        dec5 = self.dec5(torch.cat([center, conv5], dim=1))
        dec5 = nn.functional.interpolate(dec5, scale_factor=2, mode='bilinear', align_corners=True)
        
        dec4 = self.dec4(torch.cat([dec5, conv4], dim=1))
        dec4 = nn.functional.interpolate(dec4, scale_factor=2, mode='bilinear', align_corners=True)
        
        dec3 = self.dec3(torch.cat([dec4, conv3], dim=1))
        dec3 = nn.functional.interpolate(dec3, scale_factor=2, mode='bilinear', align_corners=True)
        
        dec2 = self.dec2(torch.cat([dec3, conv2], dim=1))
        dec2 = nn.functional.interpolate(dec2, scale_factor=2, mode='bilinear', align_corners=True)
        
        dec1 = self.dec1(torch.cat([dec2, conv1], dim=1))
        
        return self.final(dec1)

In [64]:
del unet_baseline


NameError: name 'unet_baseline' is not defined

In [65]:
del model

NameError: name 'model' is not defined

In [67]:
import gc
torch.cuda.empty_cache()
gc.collect()

23509

In [68]:
model = VGGUNet(n_classes=1, pretrained=True).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

unet_baseline = UNet().to(device)
unet_params = sum(p.numel() for p in unet_baseline.parameters())
print(f"\nOriginal U-Net Parameters: {unet_params:,}")
print(f"Parameter Ratio (VGG-UNet/UNet): {total_params/unet_params:.2f}x")

del unet_baseline

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()


Total Parameters: 36,697,601
Trainable Parameters: 36,697,601

Original U-Net Parameters: 32,427,521
Parameter Ratio (VGG-UNet/UNet): 1.13x


In [69]:
del unet_baseline

NameError: name 'unet_baseline' is not defined

In [70]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4)
test_loader = DataLoader(test_dataset, batch_size=4)

In [71]:
import time

epoch_times = []
for epoch in range(1):
    start_time = time.time()
    loss = train_model(model, train_loader, optimizer, criterion, device)
    val_dice = eval_model(model, val_loader, device)
    epoch_time = time.time() - start_time
    epoch_times.append(epoch_time)
    print(f'Epoch {epoch+1}: Loss={loss:.4f}, Val Dice={val_dice:.4f}, Time={epoch_time:.2f}s')

Epoch 1: Loss=0.1202, Val Dice=0.9916, Time=153.11s


In [72]:
avg_epoch_time = sum(epoch_times) / len(epoch_times)
print(f"\nAverage time per epoch: {avg_epoch_time:.2f}s")

test_dice = eval_model(model, test_loader, device)
print(f'\nFinal Test Dice Score: {test_dice:.4f}')

print(f"VGG-U-Net Parameters: {total_params:,}")
print(f"Original U-Net Parameters: {unet_params:,}")
print(f"VGG-U-Net Test Dice: {test_dice:.4f}")
print(f"Original U-Net Test Dice: 0.9751")


Average time per epoch: 153.11s

Final Test Dice Score: 0.9914
VGG-U-Net Parameters: 36,697,601
Original U-Net Parameters: 32,427,521
VGG-U-Net Test Dice: 0.9914
Original U-Net Test Dice: 0.9751


- Unet Performance
    - Epoch 3: Loss=0.0297, Val Dice=0.9757, Final Test Dice Score: 0.9751
    
- VGGUnet Performance
    - Dice Score: 0.9914

With even one epoch, and multiple reductions so as to fit within 8 gb vram, outperforms standard unet.

---